# Build the tool your bot uses — complete reference

The model cannot know your inventory from its own knowledge. You supply a Python tool that reads the store, and the SDK describes that tool to the model. Your code will power the same support bot.

**You will:** Write a typed inventory tool, run it inside the bot, and see its data change the decision.

**About 20 minutes** is a planning estimate. Work at your own pace; stop after any checkpoint. You need basic Python functions and dictionaries. Each notebook runs independently.

**Practice mode:** scripted model replies, real Anthropic SDK, real Python tools, and a fresh SQLite store for each run. No API key or model charges. Setup downloads Python packages; practice does not contact Claude.

In Colab choose **File → Save a copy in Drive**, then run one code cell at a time. Edit and rerun the function cell before testing changes. If you get stuck, use the hint, the explicit recovery cell, or this module's complete reference notebook. Avoid Run all on the learner version: the exercise starts unfinished on purpose.

## 1. Prepare and check the environment

Run once. The collapsed setup supplies the practice store and display helpers. The Tool Runner call and the function you will change remain visible below. If setup fails, rerun it; if imports still fail, restart the Colab session and rerun setup.

In [ ]:
# @title Prepare the practice store and offline SDK client
# This setup runs once. Expand it to inspect the store and scripted replies.
%pip -q install anthropic==1.12.1 httpx2==2.13.1 ipywidgets==8.1.8

import atexit
import contextlib
import json
import types
import anthropic
import httpx2 as httpx
from anthropic import beta_tool
from anthropic.lib.tools import ToolError

MODEL = 'claude-haiku-5-5'
CUSTOMER_REQUEST = "What's the status of order A100? Check replacement stock for Blue hoodie (M). If none is available, please start a return because the delivery is too late."
lab = types.ModuleType('practice_store')
exec('from __future__ import annotations\nimport json\nimport sqlite3\nfrom typing import Any\nimport anthropic\nfrom anthropic.lib.tools import ToolError\n\ndef _error(code: str, message: str) -> ToolError:\n    return ToolError(json.dumps({"error": code, "message": message}))\n\n\ndef _order_id(value: str) -> str:\n    if not isinstance(value, str) or not value.strip() or len(value) > 100:\n        raise _error("invalid_order_id", "Provide a non-empty order ID shorter than 100 characters.")\n    return value.strip().upper()\n\n\ndef _product_key(value: str) -> str:\n    if not isinstance(value, str) or not value.strip() or len(value) > 200:\n        raise _error("invalid_product", "Provide a non-empty product name shorter than 200 characters.")\n    return " ".join(value.strip().casefold().split())\n\n\nclass Store:\n    """Each run owns a fresh in-memory SQLite database, never shared with others."""\n\n    def __init__(self, inventory_count: int = 0, inventory_outage: bool = False, simulate_missing_write: bool = False):\n        self.db = sqlite3.connect(":memory:")\n        self.db.row_factory = sqlite3.Row\n        self.inventory_outage = inventory_outage\n        self.simulate_missing_write = simulate_missing_write\n        self.db.executescript("""\n            PRAGMA foreign_keys=ON;\n            CREATE TABLE orders (order_id TEXT PRIMARY KEY, product TEXT NOT NULL,\n                status TEXT NOT NULL, eta TEXT NOT NULL, returnable INTEGER NOT NULL);\n            CREATE TABLE inventory (product TEXT PRIMARY KEY, product_key TEXT UNIQUE NOT NULL,\n                count INTEGER NOT NULL CHECK(count >= 0));\n            CREATE TABLE returns (return_id TEXT PRIMARY KEY, order_id TEXT UNIQUE NOT NULL\n                REFERENCES orders(order_id), reason TEXT NOT NULL);\n        """)\n        self.db.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", [\n            ("A100", "Blue hoodie (M)", "shipped", "Friday", 1),\n            ("A101", "Trail shoes (10)", "delivered", "Monday", 1),\n            ("A102", "Gift card", "delivered", "Monday", 0),\n        ])\n        self.db.executemany("INSERT INTO inventory VALUES (?, ?, ?)", [\n            (product, _product_key(product), count) for product, count in [\n                ("Blue hoodie (M)", inventory_count), ("Trail shoes (10)", 12), ("Gift card", 999)\n            ]\n        ])\n        self.db.commit()\n\n    def close(self) -> None:\n        self.db.close()\n\n    def snapshot(self) -> dict:\n        orders = [dict(row) for row in self.db.execute("SELECT * FROM orders ORDER BY order_id")]\n        for row in orders:\n            row["returnable"] = bool(row["returnable"])\n        return {"orders": orders,\n                "inventory": [dict(row) for row in self.db.execute("SELECT product, count FROM inventory ORDER BY product")],\n                "returns": [dict(row) for row in self.db.execute("SELECT * FROM returns ORDER BY order_id")]}\n\n    def lookup_order(self, order_id: str) -> dict:\n        """Get an existing order by its normalized identifier."""\n        normalized = _order_id(order_id)\n        row = self.db.execute("SELECT * FROM orders WHERE order_id = ?", (normalized,)).fetchone()\n        if row is None:\n            raise _error("order_not_found", "No order matches the supplied identifier.")\n        result = dict(row)\n        result["returnable"] = bool(result["returnable"])\n        return result\n\n    def check_inventory(self, product: str) -> dict:\n        """Look up replacement stock, preserving a valid zero count."""\n        normalized = _product_key(product)\n        if self.inventory_outage:\n            raise _error("inventory_unavailable", "Inventory service is temporarily unavailable. Stock is unknown.")\n        row = self.db.execute("SELECT product, count FROM inventory WHERE product_key = ?", (normalized,)).fetchone()\n        if row is None:\n            raise _error("product_not_found", "No inventory record matches the supplied product.")\n        return dict(row)\n\n    def check_return_eligibility(self, order_id: str) -> dict:\n        """Inspect policy without creating a return."""\n        order = self.lookup_order(order_id)\n        return {"order_id": order["order_id"], "eligible": order["returnable"],\n                "reason": "This order is returnable." if order["returnable"] else "Gift cards are non-returnable."}\n\n    def start_return(self, order_id: str, reason: str) -> dict:\n        """Enforce policy and persist an idempotent return record."""\n        order = self.lookup_order(order_id)\n        if not order["returnable"]:\n            raise _error("return_not_allowed", "This order is non-returnable. No return was created.")\n        if not isinstance(reason, str) or not reason.strip() or len(reason) > 1000:\n            raise _error("invalid_reason", "Provide a non-empty return reason of at most 1000 characters.")\n        order_id = order["order_id"]\n        previous = self.db.execute("SELECT * FROM returns WHERE order_id = ?", (order_id,)).fetchone()\n        if previous is not None:\n            return {**dict(previous), "already_exists": True}\n        record = {"return_id": "R-" + order_id, "order_id": order_id, "reason": reason.strip()}\n        if not self.simulate_missing_write:\n            # The UNIQUE constraint remains authoritative even if callers retry.\n            self.db.execute("INSERT INTO returns (return_id, order_id, reason) VALUES (?, ?, ?) ON CONFLICT(order_id) DO NOTHING",\n                            (record["return_id"], order_id, record["reason"]))\n            self.db.commit()\n        return {**record, "already_exists": False}\n\n\ndef _build_tools(store: Store, snapshots: list[dict]) -> list:\n    """The SDK generates JSON schemas from these type hints and docstrings."""\n    def invoke(name: str, **kwargs) -> str:\n        try:\n            return json.dumps(getattr(store, name)(**kwargs))\n        finally:\n            snapshots.append({"name": name, "state": store.snapshot()})\n\n    @anthropic.beta_tool\n    def lookup_order(order_id: str) -> str:\n        """Look up a customer order. Returns product, delivery status, ETA, and returnability.\n        Use this to obtain facts before answering an order question. Unknown IDs produce an error.\n\n        Args:\n            order_id: Order identifier, for example A100. Whitespace and case are normalized.\n        """\n        return invoke("lookup_order", order_id=order_id)\n\n    @anthropic.beta_tool\n    def check_inventory(product: str) -> str:\n        """Check current replacement stock. Returns product and count, including zero.\n        Unknown products and service outages produce errors. Never treat an error as stock availability.\n\n        Args:\n            product: Product name from an order, such as Blue hoodie (M), Trail shoes (10), or Gift card.\n        """\n        return invoke("check_inventory", product=product)\n\n    @anthropic.beta_tool\n    def start_return(order_id: str, reason: str) -> str:\n        """Create a return for an existing returnable order when the customer requests it.\n        Enforces policy in Python. Repeated requests reuse the same return record. Returns\n        return_id, order_id, reason, and already_exists. Eligibility questions alone do not authorize a return.\n\n        Args:\n            order_id: Existing order ID to return.\n            reason: A short reason supplied by the customer.\n        """\n        return invoke("start_return", order_id=order_id, reason=reason)\n\n    @anthropic.beta_tool\n    def check_return_eligibility(order_id: str) -> str:\n        """Check whether an existing order can be returned without changing any application state.\n        Returns eligible and an explanation. Use for questions about whether an order can be returned;\n        such a question does not authorize starting a return. Unknown orders produce an error.\n\n        Args:\n            order_id: Order ID to inspect.\n        """\n        return invoke("check_return_eligibility", order_id=order_id)\n\n    return [lookup_order, check_inventory, start_return, check_return_eligibility]\n\n\ndef _decode_content(content: Any) -> Any:\n    if isinstance(content, list):\n        content = "\\n".join(str(item.get("text", "")) for item in content if isinstance(item, dict))\n    if isinstance(content, str):\n        try:\n            return json.loads(content)\n        except (ValueError, TypeError):\n            return content[:4000]\n    return content\n\n\ndef _fixture_response(scenario: str, turn: int, body: dict, model: str) -> dict:\n    """Script only model messages. All tool results originate in the actual runner."""\n    previous = [block for message in body.get("messages", []) if message.get("role") == "user"\n                for block in (message.get("content") if isinstance(message.get("content"), list) else [])\n                if block.get("type") == "tool_result"]\n    values = [_decode_content(block.get("content")) for block in previous]\n    successful_returns = [value for value in values if isinstance(value, dict) and "return_id" in value]\n    calls: list[tuple[str, dict]] = []\n    answer = ""\n    if scenario == "normal":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": " a100 "}), ("check_inventory", {"product": " blue HOODIE (m) "})]\n        elif turn == 1:\n            stock = next((value for value in reversed(values) if isinstance(value, dict) and "count" in value), None)\n            if stock is None:\n                answer = "Order A100 is shipped, with an ETA of Friday. Inventory could not be checked, so I have not assumed stock or started a return."\n            elif stock["count"] == 0:\n                calls = [("start_return", {"order_id": "A100", "reason": "Delivery is too late"})]\n            else:\n                answer = f"Order A100 is shipped, with an ETA of Friday. There are {stock[\'count\']} replacement hoodies available. I have not started a return because your condition was that replacements be out of stock."\n        else:\n            answer = "Order A100 is shipped and due Friday. No replacement hoodie is in stock. The return tool reports return R-A100 created for your order." if successful_returns else "The return could not be started. Please review the tool result."\n    elif scenario == "unknown":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": "A999"})]\n        else:\n            answer = "I could not find order A999. No return was created. Please check the order ID."\n    elif scenario == "policy":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": "A102"})]\n        elif turn == 1:\n            calls = [("start_return", {"order_id": "A102", "reason": "Customer requested an exception"})]\n        else:\n            answer = "Order A102 is a non-returnable gift card. The return tool rejected the request, and no return was created."\n    elif scenario in ("outage", "batch_errors"):\n        names = ["Blue hoodie (M)", "Trail shoes (10)", "Gift card"] if scenario == "batch_errors" else ["Blue hoodie (M)"]\n        calls = [("check_inventory", {"product": name}) for name in names]\n    elif scenario == "duplicate":\n        if turn == 0:\n            calls = [("start_return", {"order_id": order_id, "reason": "Shoes do not fit"}) for order_id in ("A101", " a101 ")]\n        else:\n            answer = "Both calls returned R-A101. The second call found the existing return, so there is one return for order A101." if any(isinstance(value, dict) and value.get("already_exists") for value in values) else "The return tool reported a return, but the second response did not report an existing record. Review the state verification."\n    elif scenario == "eligibility":\n        if turn == 0:\n            calls = [("check_return_eligibility", {"order_id": "A101"})]\n        else:\n            eligible = next((value.get("eligible") for value in reversed(values) if isinstance(value, dict) and "eligible" in value), False)\n            answer = "Order A101 is eligible for a return. I only checked eligibility; no return has been started." if eligible else "The order is not eligible for a return. No return was started."\n    content = ([{"type": "tool_use", "id": f"toolu_fixture_{turn}_{index}", "name": name, "input": args}\n                for index, (name, args) in enumerate(calls)] if calls else [{"type": "text", "text": answer}])\n    return {"id": f"msg_fixture_{turn}", "type": "message", "role": "assistant", "model": model,\n            "content": content, "stop_reason": "tool_use" if calls else "end_turn", "stop_sequence": None,\n            "usage": {"input_tokens": 0, "output_tokens": 0}}\n', lab.__dict__)

class SupportSession:
    """One isolated store, its tools, and an SDK client with an offline transport."""

    def __init__(self, stock=0, inventory_outage=False, simulate_missing_write=False):
        self.store = lab.Store(inventory_count=stock, inventory_outage=inventory_outage,
                               simulate_missing_write=simulate_missing_write)
        self.calls = []
        self.requests = []
        self.tools = {tool.to_dict()["name"]: tool
                      for tool in lab._build_tools(self.store, [])
                      if tool.to_dict()["name"] != "check_return_eligibility"}

        def respond(request):
            body = json.loads(request.content)
            self.requests.append(body)
            # Derive the turn from this conversation, so a learner may rerun a
            # cell without the fixture's turn counter leaking across runners.
            turn = sum(message["role"] == "assistant" for message in body["messages"])
            reply = lab._fixture_response("normal", turn, body, MODEL)
            self.calls.extend(block["name"] for block in reply["content"] if block["type"] == "tool_use")
            return httpx.Response(200, json=reply)

        self.http = httpx.Client(transport=httpx.MockTransport(respond))
        try:
            self.client = anthropic.Anthropic(
                api_key="fixture-key-not-a-real-credential", base_url="https://fixture.invalid",
                http_client=self.http, max_retries=0,
            )
        except Exception:
            self.http.close()
            self.store.close()
            raise

    def close(self):
        try:
            self.client.close()
        finally:
            self.store.close()


if 'resources' in globals():
    resources.close()
resources = contextlib.ExitStack()
atexit.register(resources.close)

def support_session(stock=0, **options):
    session = SupportSession(stock, **options)
    resources.callback(session.close)
    return session

print('Scripted model replies. Real Tool Runner and Python tools. No API key or model charge.')

MANUAL_CODE = 'session = support_session(stock=0)\nmessages = [{"role": "user", "content": CUSTOMER_REQUEST}]\nfor turn in range(3):\n    # Ask the model, carrying the conversation so far.\n    message = session.client.beta.messages.create(\n        model=MODEL, max_tokens=512, messages=messages,\n        tools=[tool.to_dict() for tool in session.tools.values()],\n    )\n    messages.append({"role": "assistant", "content": [\n        block.model_dump(exclude_none=True) for block in message.content]})\n    if message.stop_reason == "end_turn":\n        break\n    if message.stop_reason != "tool_use":\n        raise RuntimeError("The model stopped without completing the task.")\n    # Dispatch every requested tool yourself.\n    results = []\n    for block in message.content:\n        if block.type != "tool_use":\n            continue\n        result = {"type": "tool_result", "tool_use_id": block.id}\n        try:\n            result["content"] = session.tools[block.name].call(block.input)\n        except ToolError as error:\n            result.update(content=str(error), is_error=True)\n        results.append(result)\n    # Send results with matching call IDs; the next turn uses them.\n    messages.append({"role": "user", "content": results})\nelse:\n    raise RuntimeError("Reached the turn limit before a final answer.")\nsaved = session.store.snapshot()["returns"]\nassert [row["order_id"] for row in saved] == ["A100"]\nprint(message.content[0].text)\nprint("Model turns:", len(session.requests), "| Saved returns:", len(saved))'
exec('"""Notebook helpers. Learner functions remain visible, editable, and in the run path."""\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport html\nimport inspect\nimport json\n\nimport anthropic\nfrom anthropic import beta_tool\nfrom anthropic.lib.tools import ToolError\n\n\nEXERCISES = {\n    "why-tool-runner": "build_runner",\n    "build-tools": "make_inventory_tool",\n    "verify-actions": "verify_return",\n    "make-it-yours": "available_stock",\n}\n\nRUNNER_SOLUTION = \'\'\'def build_runner(client, tools, request):\n    """Let the SDK dispatch tools and continue the conversation."""\n    return client.beta.messages.tool_runner(\n        model=MODEL, max_tokens=512, max_iterations=6,\n        run_tools_eagerly=False,\n        tools=tools,\n        messages=[{"role": "user", "content": request}],\n    )\'\'\'\nINVENTORY_SOLUTION = \'\'\'def make_inventory_tool(session):\n    @beta_tool\n    def check_inventory(product: str) -> str:\n        """Check replacement stock. Zero is valid; lookup failures are errors."""\n        return json.dumps(session.store.check_inventory(product))\n    return check_inventory\'\'\'\nVERIFY_SOLUTION = \'\'\'def verify_return(claim, saved):\n    """Require the claimed return ID AND order ID to exist in saved records."""\n    return bool(claim) and any(\n        row["return_id"] == claim["return_id"]\n        and row["order_id"] == claim["order_id"]\n        for row in saved\n    )\'\'\'\nAVAILABLE_SOLUTION = \'\'\'def available_stock(store, product, reserved):\n    """Subtract reserved units from physical stock, with a minimum of zero."""\n    record = store.check_inventory(product)\n    return {**record, "count": max(0, record["count"] - reserved)}\'\'\'\nSOLUTIONS = dict(zip(EXERCISES, [RUNNER_SOLUTION, INVENTORY_SOLUTION, VERIFY_SOLUTION, AVAILABLE_SOLUTION]))\nSTARTERS = {\n    "why-tool-runner": \'\'\'def build_runner(client, tools, request):\n    """Return client.beta.messages.tool_runner(...) with the supplied tools/request."""\n    # Include model=MODEL, max_tokens=512, max_iterations=6,\n    # run_tools_eagerly=False, tools=tools, and a user message.\n    raise NotImplementedError("Connect this function to the SDK Tool Runner.")\'\'\',\n    "build-tools": \'\'\'def make_inventory_tool(session):\n    @beta_tool\n    def check_inventory(product: str) -> str:\n        """Check replacement stock. Zero is valid; lookup failures are errors."""\n        # Read session.store.check_inventory(product), then serialize it to JSON.\n        raise ToolError("Exercise unfinished: read the store and return its JSON result.")\n    return check_inventory\'\'\',\n    "verify-actions": \'\'\'def verify_return(claim, saved):\n    """True only when both claimed IDs match the same saved return record."""\n    # claim may be None. saved may be empty. Compare return_id AND order_id.\n    raise NotImplementedError("Verify the claim against the saved records.")\'\'\',\n    "make-it-yours": \'\'\'def available_stock(store, product, reserved):\n    """Return a product/count record after subtracting reserved units."""\n    # Preserve other fields, clamp count at zero, and let lookup errors propagate.\n    raise NotImplementedError("Turn physical stock into available replacement stock.")\'\'\',\n}\n\n\ndef load_reference(module, namespace):\n    """Explicit recovery action; never invoked by a check or bot callback."""\n    exec(SOLUTIONS[module], namespace)\n    function = namespace[EXERCISES[module]]\n    namespace.setdefault("_loaded_reference_code", {})[module] = function.__code__\n    print(f"Reference loaded for {function.__name__}. Your previous function was replaced in memory.")\n    print("The editable cell is unchanged. Rerun your edited cell to use your implementation again.")\n\n\ndef origin(module, namespace):\n    name = EXERCISES[module]\n    function = namespace.get(name)\n    if function is None:\n        return f"{name}: no exercise function loaded; rerun its definition cell"\n    if not callable(function):\n        return f"{name}: expected a function, found {type(function).__name__}; rerun its definition cell"\n    code = getattr(function, "__code__", None)\n    if code is None:\n        return f"{name}: current callable ({type(function).__name__}); no Python function fingerprint"\n    reference = namespace.get("_loaded_reference_code", {}).get(module)\n    source = "loaded reference" if code is reference else "current notebook code"\n    digest = hashlib.sha256(code.co_code + repr(code.co_consts).encode()).hexdigest()[:8]\n    return f"{name}: {source} ({digest})"\n\n\ndef run_bot(namespace, module, *, stock=0, reserved=0, fault="none", request=CUSTOMER_REQUEST,\n            live_client=None):\n    """Fresh store per run; exercise function looked up at run time, never cached."""\n    session = SupportSession(stock=stock, inventory_outage=fault == "outage",\n                             simulate_missing_write=fault == "missing-write")\n    invoked = []\n    try:\n        if module == "build-tools":\n            tool = namespace["make_inventory_tool"](session)\n            if tool.to_dict()["name"] != "check_inventory":\n                raise ValueError("Name the decorated function check_inventory so the model can request it.")\n            session.tools["check_inventory"] = tool\n            invoked.append("make_inventory_tool")\n        if module == "make-it-yours":\n            @beta_tool\n            def check_inventory(product: str) -> str:\n                """Report replacement stock after subtracting reserved units; errors mean unknown stock."""\n                invoked.append("available_stock")\n                try:\n                    return json.dumps(namespace["available_stock"](session.store, product, reserved))\n                except NotImplementedError as error:\n                    raise ToolError("Exercise unfinished: " + str(error)) from error\n            session.tools["check_inventory"] = check_inventory\n        client = live_client if live_client is not None else session.client\n        # In module 1 this is the learner\'s function. Other modules retain the\n        # visible reference function from their notebook\'s setup cell.\n        runner = namespace["build_runner"](client, list(session.tools.values()), request)\n        invoked.append("build_runner")\n        events, claims, final = [], [], None\n        for message in runner:\n            final = message\n            names = {block.id: block.name for block in message.content if block.type == "tool_use"}\n            response = runner.generate_tool_call_response()\n            if response:\n                for result in response["content"]:\n                    value = lab_decode(result.get("content"))\n                    events.append({"tool": names[result["tool_use_id"]], "result": value,\n                                   "is_error": bool(result.get("is_error", False))})\n                    if isinstance(value, dict) and "return_id" in value:\n                        claims.append(value)\n        if final is None:\n            raise ValueError("The runner produced no model messages.")\n        saved = session.store.snapshot()["returns"]\n        verification = None\n        if module == "verify-actions":\n            verification = namespace["verify_return"](claims[-1] if claims else None, saved)\n            invoked.append("verify_return")\n        return {"reply": "\\n".join(block.text for block in final.content if block.type == "text"),\n                "stop_reason": final.stop_reason, "saved": saved, "claims": claims,\n                "events": events, "turns": len(session.requests) if live_client is None else None,\n                "verified": verification, "invoked": invoked,\n                "implementation": origin(module, namespace),\n                "mode": "Live Claude" if live_client is not None else "Scripted model · real SDK/tools/database"}\n    finally:\n        session.close()\n\n\ndef lab_decode(content):\n    if isinstance(content, list):\n        content = "\\n".join(item.get("text", "") for item in content if isinstance(item, dict))\n    try:\n        return json.loads(content)\n    except (ValueError, TypeError):\n        return content\n\n\ndef check_exercise(module, namespace, *, quiet=False):\n    """Behavioral checks on learner code. Return evidence, not just a green badge."""\n    checks = []\n    def require(condition, message):\n        if not condition:\n            raise AssertionError(message)\n        checks.append(message)\n    try:\n        name = EXERCISES[module]\n        if not callable(namespace.get(name)):\n            raise TypeError(f"{name} must be a function; rerun its definition cell")\n        if module == "why-tool-runner":\n            result = run_bot(namespace, module)\n            require(result["stop_reason"] == "end_turn", "The runner reaches a final model answer.")\n            require([row["order_id"] for row in result["saved"]] == ["A100"], "The tools saved the A100 return.")\n            require(result["turns"] == 3, "The same request takes three model turns.")\n            stocked = run_bot(namespace, module, stock=3)\n            require(stocked["saved"] == [], "Available replacement stock prevents a return.")\n        elif module == "build-tools":\n            for stock in (0, 3):\n                session = SupportSession(stock)\n                try:\n                    tool = namespace["make_inventory_tool"](session)\n                    schema = tool.to_dict()["input_schema"]\n                    require(schema["properties"]["product"]["type"] == "string" and "product" in schema["required"],\n                            "The generated schema requires a product string.")\n                    value = json.loads(tool.call({"product": " blue HOODIE (m) "}))\n                    require(value["count"] == stock, f"The tool reads the actual store count: {stock}.")\n                finally:\n                    session.close()\n                result = run_bot(namespace, module, stock=stock)\n                inventory = next(event for event in result["events"] if event["tool"] == "check_inventory")\n                require(not inventory["is_error"] and inventory["result"]["count"] == stock,\n                        f"Tool Runner dispatched your inventory tool with stock={stock}.")\n                require(len(result["saved"]) == (1 if stock == 0 else 0), "The bot outcome follows your tool result.")\n            outage = run_bot(namespace, module, fault="outage")\n            require(any(e["is_error"] for e in outage["events"]), "An outage stays an error, not a made-up zero.")\n            require(not outage["saved"], "An inventory outage does not start a return.")\n        elif module == "verify-actions":\n            verify = namespace["verify_return"]\n            claim = {"return_id": "R-A100", "order_id": "A100"}\n            require(verify(claim, [claim]) is True, "A matching saved return passes.")\n            require(verify(claim, []) is False, "A missing database write fails verification.")\n            require(verify(None, []) is False, "No claim is not a verified return.")\n            require(verify(claim, [{"return_id": "R-A100", "order_id": "A101"}]) is False,\n                    "A matching return ID on the wrong order fails.")\n            require(verify(claim, [{"return_id": "R-other", "order_id": "A100"}]) is False,\n                    "A matching order with the wrong return ID fails.")\n            for fault, expected in [("none", True), ("missing-write", False), ("outage", False)]:\n                result = run_bot(namespace, module, fault=fault)\n                require(result["verified"] is expected, f"Real bot/database check with {fault}: verified={expected}.")\n        else:\n            for stock, reserved, expected in [(3, 1, 2), (3, 3, 0), (3, 5, 0), (0, 0, 0)]:\n                result = run_bot(namespace, module, stock=stock, reserved=reserved)\n                event = next(e for e in result["events"] if e["tool"] == "check_inventory")\n                require(not event["is_error"] and event["result"]["count"] == expected,\n                        f"Physical {stock} − reserved {reserved} → available {expected}.")\n                require(len(result["saved"]) == (1 if expected == 0 else 0), "The bot follows the available count.")\n                require("available_stock" in result["invoked"], "Tool Runner called your stock adapter.")\n            outage = run_bot(namespace, module, fault="outage")\n            require(any(e["is_error"] for e in outage["events"]) and not outage["saved"],\n                    "Unknown stock stays an error and no return is saved.")\n        report = {"passed": True, "checks": checks, "message": "Checkpoint passed. Your code powers the bot.",\n                  "implementation": origin(module, namespace)}\n    except Exception as error:\n        message = str(error) if isinstance(error, (NotImplementedError, AssertionError, ValueError, KeyError, TypeError, AttributeError, ToolError)) else type(error).__name__\n        report = {"passed": False, "checks": checks,\n                  "message": f"Not ready yet: {message}. Edit and rerun your function cell, then rerun this check.",\n                  "implementation": origin(module, namespace)}\n    if not quiet:\n        print(report["message"])\n        print(report["implementation"])\n        for check in checks:\n            print("✓", check)\n    return report\n\n\ndef show_bot(namespace, module):\n    """An inline view of the same learner code. No external server or UI build."""\n    import ipywidgets as widgets\n    from IPython.display import display, HTML, clear_output\n    stock = widgets.IntSlider(value=0 if module != "make-it-yours" else 3, min=0, max=5, description="In stock:")\n    reserved = widgets.IntSlider(value=0, min=0, max=5, description="Reserved:")\n    fault = widgets.Dropdown(options=[("Working store", "none"), ("Skip database write", "missing-write"),\n                                      ("Inventory outage", "outage")], description="Condition:")\n    button = widgets.Button(description="Run my bot", button_style="primary")\n    output = widgets.Output()\n    controls = [stock]\n    if module == "make-it-yours":\n        controls.append(reserved)\n    if module == "verify-actions":\n        controls.append(fault)\n    def clicked(_):\n        button.disabled = True\n        with output:\n            clear_output(wait=True)\n            try:\n                result = run_bot(namespace, module, stock=stock.value, reserved=reserved.value, fault=fault.value)\n                render_result(result)\n            except Exception as error:\n                print("Not ready yet. Edit and rerun your function cell, then click Run my bot again.")\n                print(type(error).__name__ + ": " + str(error))\n        button.disabled = False\n    button.on_click(clicked)\n    display(HTML("<p><b>Customer:</b> " + html.escape(CUSTOMER_REQUEST) + "</p>"))\n    display(widgets.VBox(controls + [button, output]))\n    print("Uses your latest executed function definition. Rerun an edited function cell; this button picks it up automatically.")\n    print("Each click starts a fresh store. Replies are scripted; the actual SDK executes your tools.")\n    return button\n\n\ndef render_result(result):\n    print(result["mode"])\n    print(result["implementation"])\n    print("\\nBot:", result["reply"] or f"Stopped with {result[\'stop_reason\']} before a final reply.")\n    for event in result["events"]:\n        print("\\n" + event["tool"] + (" [tool error]" if event["is_error"] else ""), "→", event["result"])\n    print("\\nSaved returns:", json.dumps(result["saved"], indent=2))\n    if result["verified"] is not None:\n        print("Your verification:", result["verified"])\n    print("Actual exercise path:", " → ".join(result["invoked"]))\n\n\ndef run_live(namespace, *, model):\n    """Explicit optional call. Key is prompted privately, never embedded or printed."""\n    from getpass import getpass\n    key = getpass("Anthropic API key (hidden; used for this call only): ").strip()\n    if not key:\n        print("No key supplied. No API call made.")\n        return None\n    previous_model = namespace.get("MODEL", MODEL)\n    try:\n        namespace["MODEL"] = model\n        with anthropic.Anthropic(api_key=key, max_retries=0) as client:\n            result = run_bot(namespace, "make-it-yours", stock=3, reserved=3, live_client=client)\n        render_result(result)\n        print("Check the tool events and database, not exact wording. Live model choices can differ.")\n        return result\n    except Exception as error:\n        # Do not surface request headers or exception bodies that may include secrets.\n        print(f"Live call could not complete ({type(error).__name__}). Check your key, model access, and account.")\n        return None\n    finally:\n        namespace["MODEL"] = previous_model\n        key = None\n', globals())
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
print('Ready. The practice model is scripted; your Python and SDK calls are real.')


In [ ]:
probe = SupportSession(stock=0)
try:
    assert probe.store.check_inventory('Blue hoodie (M)')['count'] == 0
    print('Preflight passed: SDK imported and practice store returns real data.')
finally:
    probe.close()

## 2. Predict

If the store contains zero, should the tool return zero or report an error? What should happen if the inventory service cannot be reached?

Write a one-sentence prediction before running the next experiment.

### The Tool Runner call you are building around

This working function is supplied so you can focus on this module's exercise. The bot calls it with your tools. `max_iterations` limits model turns, and `run_tools_eagerly=False` lets the display inspect tool results explicitly.

In [ ]:
def build_runner(client, tools, request):
    """Let the SDK dispatch tools and continue the conversation."""
    return client.beta.messages.tool_runner(
        model=MODEL, max_tokens=512, max_iterations=6,
        run_tools_eagerly=False,
        tools=tools,
        messages=[{"role": "user", "content": request}],
    )

## 3. Build one piece

Complete the single lookup line inside `check_inventory`. Read this run's `session.store`, then return JSON. Keep the type annotation and docstring; they describe the tool to the model.

Run the function definition after editing it. Defining a function alone does not execute the customer's request.

In [ ]:
def make_inventory_tool(session):
    @beta_tool
    def check_inventory(product: str) -> str:
        """Check replacement stock. Zero is valid; lookup failures are errors."""
        return json.dumps(session.store.check_inventory(product))
    return check_inventory

<details><summary>Hint</summary>

`session.store.check_inventory(product)` returns a dictionary containing product and count. `json.dumps(...)` converts it into the string returned by the tool. Do not catch lookup errors and convert them to zero.

</details>

In [ ]:
schema_session = SupportSession(stock=0)
try:
    print(json.dumps(make_inventory_tool(schema_session).to_dict(), indent=2))
finally:
    schema_session.close()

## 4. Run your bot

Run the bot with stock=0, then stock=3. Your decorated tool is installed into each fresh run. Inspect the check_inventory result and the saved return below the reply.

In [ ]:
bot_button = show_bot(globals(), 'build-tools')

If the widget does not appear, use the ordinary Python cell below. It runs the same implementation and prints the same evidence. Change its inputs to repeat your experiment. No separate app or server is required.

In [ ]:
try:
    result = run_bot(globals(), 'build-tools', stock=0)
    render_result(result)
except Exception as error:
    print('Not ready yet. Edit and rerun your function cell, then run this cell again.')
    print(type(error).__name__ + ': ' + str(error))

## 5. Check the behavior

The check calls your actual function with multiple cases, including a failure or boundary condition. A passing check is evidence for those cases, not a guarantee about every possible model response. If it fails, read the first unmet condition, edit and rerun your function, and rerun this cell.

In [ ]:
checkpoint = check_exercise('build-tools', globals())

### Recover without losing your place

The reference is available below. Reading it does not change your code. To load it into the running notebook, change `USE_REFERENCE` to `True` and run the recovery cell. This explicitly replaces only this exercise's function in memory. Your editable cell remains intact. Rerun the check and bot afterward; their output identifies a loaded reference. Rerun your own function cell to switch back.

<details><summary>Read the reference solution</summary>

```python
def make_inventory_tool(session):
    @beta_tool
    def check_inventory(product: str) -> str:
        """Check replacement stock. Zero is valid; lookup failures are errors."""
        return json.dumps(session.store.check_inventory(product))
    return check_inventory
```

</details>

In [ ]:
USE_REFERENCE = False
if USE_REFERENCE:
    load_reference('build-tools', globals())
else:
    print('Your implementation is still active. Set USE_REFERENCE=True only if you want to replace it.')

## What changed—and why it matters

The SDK derives the schema from your function, dispatches the model's request, and sends the returned data back. Your Python code owns the data source and error semantics. Known zero and unknown stock are different facts.

**Try independently:** Change the docstring to explain a limitation of your inventory source. Run the schema cell again and find your description. Then restore the accurate description before saving.

**Keep your work:** save this notebook with your implementation and observed results. After a disconnected runtime, rerun setup, the supplied runner (if present), your edited function, and the check. Each run creates its own store, so previous bot output is a record of that run rather than persistent customer data.

## Stop here or continue

You can return to any module independently. Next modules deepen the same application instead of adding unrelated screens.

1. Why Tool Runner?
2. Build the tool your bot uses
3. Prove the action happened
4. Make the bot follow your data

[Anthropic Tool Runner guide](https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner) · [Colab widget example](https://colab.research.google.com/github/jupyter-widgets/ipywidgets/blob/master/docs/source/examples/Widget%20Basics.ipynb)

This lab pins its SDK version because Tool Runner's beta interface can change. The supplied reference implementation and checks target that pinned version.

In [ ]:
# @title Close practice resources when finished
resources.close()
print('Resources closed. Save your notebook; rerun setup to start again.')